In [18]:
import pandas as pd
import re
from collections import Counter
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

CSV_FILE_PATH = 'DevESS-AI-projectPhising_output.csv'
LABELED_OUTPUT_PATH = 'phishguard_training_data_ai_labeled.csv'
TFIDF_VECTORIZER_PATH = 'tfidf_vectorizer.pkl'
DBSCAN_MODEL_PATH = 'dbscan_brand_model.pkl'
BRAND_MAPPING_OUTPUT_PATH = 'dbscan_brand_clusters.csv'

#marka etiketlenmesinde göz ardı edilecek kelimeler
NOISE_WORDS = {
    'online', 'giris', 'login', 'secure', 'bank', 'hesap', 'sifre','kredi', 'ihtiyac', 'guvenlik', 'web', 'app', 'site', 'mobil','com', 'net', 'org', 'info', 'store', 'biz', 'co', 'one', 'xyz', 'auth','link', 'page', 'indirim', 'firsat', 'portal', 'service','support', 'verify', 'update', 'security', 'kampanya', 'user', 'sign','account', 'group', 'data', 'click', 'life', 'islemler','urun', 'server','home', 'post', 'view', 'download', 'help','token', 'verification', 'alert', 'track', 'event','system', 'forms','platform', 'mobile', 'customer', 'wallet', 'push', 'social', 'islem', 'ilan','sorgu', 'host', 'analyze','tr', 'ilanlar', 'basvuru','alisveris', 'yatirim', 'sube', 'ozel', 'yeni', 'sepet','hizli', 'bireysel','finans', 'guzel', 'glitch', 'rest', 'sales', 'dns'}
df = None

In [3]:
def preprocess_url_for_embedding(url): # purpose: clean the URL for DBSCAN

    if pd.isna(url): return ""  # handle NaN (not a number) values
    url = str(url).lower()

    url = re.sub(r'https?://|www\.|/.*|\.(?:com|net|org|xyz|one|store|blog|tr|biz|co)|[?&=@_\-]', ' ', url)
    url = re.sub(r'\s+', ' ', url).strip()

    words = [word for word in re.findall(r'\b\w+\b', url) if word not in NOISE_WORDS and len(word) > 2]   # remove noise words and short words
    return ' '.join(words) # return the cleaned text

def identify_cluster_brand(cluster_data):   # purpose: for each cluster that DBSCAN identifies, identify the brand name

    full_text = ' '.join(cluster_data['PROCESSED_TEXT'])   # kümedeki tüm urlleri birleştirip bir kelime havuzu oluşturduk
    words = [word for word in re.findall(r'\b\w+\b', full_text) if len(word) > 3]  # metin havuzunu tek tek kelimelere ayırıyoruz

    word_counts = Counter(words)  # her kelimeyi sayar

    if word_counts:
        top_words = word_counts.most_common(10)   # En sık geçen ilk 10 kelimeyi kontrol et
        for word, count in top_words:   # en spesifik kelimeyi bulmaya çalışıyoruz.
            if word not in NOISE_WORDS:
                return word # Gürültü listesinde olmayan, en sık geçen spesifik kelimeyi al.

        # Eğer ilk 10 kelimenin hepsi gürültü listesindeyse
        return 'AMBIGUOUS_BRAND_CANDIDATE'

    return 'UNKNOWN'

def starts_with_digit(url):
    return bool(re.match(r'^\d', str(url)))

def alphabetic_length(url):
    return len(re.findall(r'[a-zA-Z]', str(url)))

In [76]:
# --- Veri Yükleme dbscan için dosyaları birleştirme---
try:
    df1 = pd.read_csv('dbscan_brand_clusters.csv', header=0,usecols=[0,1], names=['URL', 'WEIGHT'])
    df2 = pd.read_csv('dbscan_brand_clusters2.csv', header=0,usecols=[0,1], names=['URL', 'WEIGHT'])
    df3 = pd.read_csv('dbscan_brand_clusters3.csv', header=0,usecols=[0,1], names=['URL', 'WEIGHT'])
    df4 = pd.read_csv('dbscan_brand_clusters4.csv', header=0,usecols=[0,1], names=['URL', 'WEIGHT'])
    df5 = pd.read_csv('dbscan_brand_clusters5.csv', header=0,usecols=[0,1], names=['URL', 'WEIGHT'])

    df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

    df = (df.groupby('URL', as_index=False).agg({'WEIGHT': 'sum'}) )   #aynı marka isimlerini bir araya topladım

    df= df.sort_values(by=['WEIGHT'], ascending=False).reset_index(drop=True)

    possible_brand= df.head(232)['URL'] #daha sonra dbscan sonucu ile birleştiricez

    df = df.iloc[232:].reset_index(drop=True)

    df.to_csv('collected_df.csv', index=False)
    print("collected df is saved to csv")


except FileNotFoundError:
    df = None

if df is not None:

    print(f"Veri yüklendi. Toplam kayıt: {len(df)}")

    # dataframe'e PROCESSED_TEXT sütunu ekle, URL'yi gereksiz karakterlerden ayır ve sütuna ekle
    df['PROCESSED_TEXT'] = df['URL'].apply(preprocess_url_for_embedding)

    # ngramrange=(1,2) ile hem tek kelimeleri hem de ikili grupları alır. En önemli 3000 kelimeye kadar
    vectorizer = TfidfVectorizer(analyzer="char",
                                 ngram_range=(3,5),
                                 max_features=200,
                                 min_df=2,
                                 max_df=0.9)
    X = vectorizer.fit_transform(
        df['PROCESSED_TEXT'])  # fit -> vektörleyici tüm veri setini okur her kelimenin nadirliğini hesaplar
    # transform -> urlleri sayısal vektöre dönüştürür

    # Vektörleyiciyi kaydet->dump
    joblib.dump(vectorizer, 'tfidf5_vectorizer.pkl')
    print(f"✅ TF-IDF Vektörleyici Kaydedildi: ")
    print(f"Oluşturulan Öznitelik Sayısı: {X.shape[1]}")

collected df is saved to csv
Veri yüklendi. Toplam kayıt: 4216
✅ TF-IDF Vektörleyici Kaydedildi: 
Oluşturulan Öznitelik Sayısı: 200


In [58]:
from sklearn.decomposition import TruncatedSVD
import hdbscan
if 'X' in locals() and df is not None:
    print("SVD uygulanıyor...")
    svd = TruncatedSVD(n_components=50, random_state=42)
    X_reduced = svd.fit_transform(X)
    print(f"SVD tamamlandı. Yeni vektör boyutu: {X_reduced.shape}")

    sample_weight = df['WEIGHT'].values.astype(float)
    sample_weight[sample_weight <= 0] = 1.0

    hdb = hdbscan.HDBSCAN(
        min_cluster_size=10,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    df['CLUSTER_ID'] = hdb.fit_predict(X_reduced)

    joblib.dump(hdb, DBSCAN_MODEL_PATH.replace("dbscan", "hdbscan"))

    n_clusters = len(set(df['CLUSTER_ID'])) - (1 if -1 in df['CLUSTER_ID'].values else 0)
    print(f"DBSCAN Model Kaydedildi: {DBSCAN_MODEL_PATH}")
    print(f"Toplam Tespit Edilen Marka Grubu Sayısı (Gürültü hariç): {n_clusters}")
    print(f"Gürültüye Atanan Kayıt Sayısı (-1): {(df['CLUSTER_ID'] == -1).sum()}")
else:
    print("HATA: DBSCAN çalıştırılamadı.")

SVD uygulanıyor...
SVD tamamlandı. Yeni vektör boyutu: (5024, 50)
DBSCAN Model Kaydedildi: dbscan_brand_model.pkl
Toplam Tespit Edilen Marka Grubu Sayısı (Gürültü hariç): 97
Gürültüye Atanan Kayıt Sayısı (-1): 1218


In [77]:
from sklearn.cluster import DBSCAN

if 'X' in locals() and df is not None:

    sample_weight = df['WEIGHT'].values.astype(float)
    sample_weight[sample_weight <= 0] = 1.0

    dbscan = DBSCAN(eps=0.25, min_samples=10, metric='cosine', n_jobs=-1)

    df['CLUSTER_ID'] = dbscan.fit_predict(X,sample_weight=sample_weight)

    joblib.dump(dbscan, DBSCAN_MODEL_PATH)

    n_clusters = len(set(df['CLUSTER_ID'])) - (1 if -1 in df['CLUSTER_ID'].values else 0)
    print(f"DBSCAN Model Kaydedildi: {DBSCAN_MODEL_PATH}")
    print(f"Toplam Tespit Edilen Marka Grubu Sayısı : {n_clusters}")
    print(f"Gürültüye Atanan Kayıt Sayısı (-1): {(df['CLUSTER_ID'] == -1).sum()}")
else:
    print("HATA: DBSCAN çalıştırılamadı.")


--- DBSCAN Kümelemesi Başlatılıyor ---
DBSCAN Model Kaydedildi: dbscan_brand_model.pkl
Toplam Tespit Edilen Marka Grubu Sayısı : 518
Gürültüye Atanan Kayıt Sayısı (-1): 657


In [78]:
if 'CLUSTER_ID' in df.columns:
    cluster_mapping = {}
    for cluster_id in df['CLUSTER_ID'].unique():
        if cluster_id == -1 :
            cluster_mapping[cluster_id] = 'NOISE'
            continue

        cluster_data = df[df['CLUSTER_ID'] == cluster_id]
        brand_name = identify_cluster_brand(cluster_data)
        if brand_name is None or len(brand_name) > 14:
            cluster_mapping[cluster_id] = 'NOISE'
        else:
            cluster_mapping[cluster_id] = brand_name

    df['TARGETED_BRAND'] = df['CLUSTER_ID'].map(cluster_mapping)

    cluster_summary = df.groupby('TARGETED_BRAND').agg(NumberOfElements=('CLUSTER_ID', 'size')).reset_index()

    noise_row = cluster_summary[cluster_summary['TARGETED_BRAND'] == 'NOISE'].copy()

    if not noise_row.empty:
        noise_row.at[noise_row.index[0], 'TARGETED_BRAND'] = 'NOISE'
        clean_summary = cluster_summary[(cluster_summary['TARGETED_BRAND'] != 'NOISE') & (cluster_summary['TARGETED_BRAND'] != 'UNKNOWN')]
        # Gürültüyü en üste taşı
        cluster_summary = pd.concat([noise_row, clean_summary], ignore_index=True)

    cluster_summary.to_csv('dbscan_brand_clustersFinale.csv', index=False)
    print(f"✅ Marka Küme Özet Raporu Kaydedildi: dbscan_brand_clusters.csv")

    df.to_csv(LABELED_OUTPUT_PATH, index=False)
    print(f"\n✅ Yapay Zeka ile Etiketlenmiş Veri Kaydedildi: {LABELED_OUTPUT_PATH}")

else:
    print("HATA: Kümeleme sonucu bulunamadı. Lütfen önceki adımları tamamlayın.")

✅ Marka Küme Özet Raporu Kaydedildi: dbscan_brand_clusters.csv

✅ Yapay Zeka ile Etiketlenmiş Veri Kaydedildi: phishguard_training_data_ai_labeled.csv


In [80]:
# possible_brand ile dbscan_brand_clustersFinale.csv dosyasini birlestir + kalite kontrol uygula
MIN_BRAND_FREQ = 2
BRAND_NOISE_BLACKLIST = {
    "noise",
    "unknown",
    "ambiguous_brand_candidate",
    "login",
    "verify",
    "secure",
    "support",
    "service",
    "update",
}


def normalize_brand_token(value):
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]", "", value)
    return value


try:
    cluster_df = pd.read_csv("dbscan_brand_clustersFinale.csv", usecols=[0], header=0)
    if "TARGETED_BRAND" not in cluster_df.columns:
        cluster_df.columns = ["TARGETED_BRAND"]
    cluster_series = cluster_df["TARGETED_BRAND"].dropna().astype(str).reset_index(drop=True)
except Exception:
    cluster_series = pd.Series(dtype=str)

if "possible_brand" not in locals():
    possible_brand = pd.Series(dtype=str)

possible_series = possible_brand.dropna().astype(str).reset_index(drop=True)
combined_series = pd.concat([cluster_series, possible_series], ignore_index=True)

normalized = combined_series.apply(normalize_brand_token)
normalized = normalized[normalized.str.len() >= 3]
normalized = normalized[~normalized.isin(BRAND_NOISE_BLACKLIST)]

brand_freq = normalized.value_counts().reset_index()
brand_freq.columns = ["BRAND_OR_URL", "FREQ"]
filtered_brand_freq = brand_freq[brand_freq["FREQ"] >= MIN_BRAND_FREQ].copy()

filtered_brand_freq[["BRAND_OR_URL"]].to_csv("phising_brands.csv", index=False)
filtered_brand_freq.to_csv("phising_brands_qc_report.csv", index=False)
print(f"✅ phising_brands.csv saved. Rows: {len(filtered_brand_freq)}")
print(f"✅ phising_brands_qc_report.csv saved. Rows: {len(filtered_brand_freq)}")


✅ phising_brands.csv saved. Rows: 544


In [1]:
import pandas as pd
import unicodedata
import re
 #handle turkish characters and normalize text
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).replace('İ', 'i').replace('I', 'ı').replace('Ş', 's').replace('Ğ', 'g').replace('Ü', 'u').replace('Ö', 'o').replace('Ç', 'c')
    text = str(text).lower()
    text = text.translate(str.maketrans({
        "ç": "c",
        "ğ": "g",
        "ı": "i",
        "ö": "o",
        "ş": "s",
        "ü": "u"
    }))
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9]", "", text)

    return text


In [2]:
df = pd.read_csv('brands.csv', encoding='iso-8859-9')

df['NORMALIZED_BRAND'] = df['Brand Name'].apply(normalize_text)

df.to_csv('brands_normalized.csv', index=False, columns=['NORMALIZED_BRAND'])



In [3]:
df2 = pd.read_csv("phising_brands.csv")
df2["BRAND_OR_URL"] = df2["BRAND_OR_URL"].astype(str).apply(normalize_text)

matches = df2[df2["BRAND_OR_URL"].isin(df["NORMALIZED_BRAND"])].drop_duplicates()

if not matches.empty:
    matches.to_csv("matched_phising_brands.csv", index=False)
    print(f"Eslesme bulundu! {len(matches)} adet marka 'matched_phising_brands.csv' dosyasina yazildi.")
else:
    print("Hicbir eslesme bulunamadi.")



Eşleşme bulundu! 44 adet marka 'matched_phising_brands.csv' dosyasına yazıldı.
